# Machine Learning Coursework: CA1 & CA2 Project
## Project Title: Real-World Stock Price Predictor & Volatility Regime Segmentation System
**Student Mode:** Solo Project  
**Submission Components:**  
- **CA1:** Problem Statement, Data Collection, Exploratory Data Analysis (EDA), Feature Engineering & Preprocessing.  
- **CA2:** Implementation of Minimum 3 Advanced ML Algorithms (Gradient Boosting, Random Forest, Support Vector Regression, K-Means Clustering), Comparative Evaluation (R², MAE, RMSE, MAPE), and Interactive Web Interface Deployment.

--- 
## Part 1: CA1 - Problem Formulation & Dataset Ingestion
Stock price movements are intrinsically non-linear, exhibiting volatility clustering, regime shifts, and asymmetric tail risks. Traditional naive linear models fail to capture deep interaction dynamics such as momentum exhaustion (overbought RSI combined with elevated Beta) and liquidity shocks. In this project, we utilize real-world historical equity data containing 2,000 records across 5 primary market sectors with engineered technical and fundamental indicators.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load real dataset
df = pd.read_csv('../dataset/stock_market_dataset.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Exploratory Data Analysis (EDA) & Summary Statistics

In [2]:
# Summary statistics
df[['close_price', 'rsi_14', 'sma_ratio', 'volatility_30d', 'beta', 'target_next_price']].describe().T

In [3]:
# Missing value analysis
print("Missing values per column:")
print(df.isnull().sum())

--- 
## Part 2: CA1 - Feature Engineering & Preprocessing Pipeline

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ['close_price', 'rsi_14', 'sma_ratio', 'volatility_30d', 'beta', 'earnings_growth_pct', 'institutional_flow_idx']
target = 'target_next_price'

X = df[features].values
y = df[target].values

# 80/20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training set: {X_train.shape[0]} samples | Testing set: {X_test.shape[0]} samples")

--- 
## Part 3: CA2 - Multi-Algorithm Implementation (4 Algorithms)
We implement four distinct algorithms exceeding the minimum 3 algorithm requirement:
1. **Gradient Boosting Regressor (GBM)** - Sequential ensemble minimizing squared error loss with shrinkage.
2. **Random Forest Regressor** - Bagging ensemble of 100 decorrelated decision trees.
3. **Support Vector Regression (SVR)** - Non-linear RBF kernel with $\varepsilon$-margin loss.
4. **K-Means Clustering** - Unsupervised market regime segmentation.

In [5]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# 1. Gradient Boosting
gbm = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
gbm.fit(X_train, y_train)
pred_gbm = gbm.predict(X_test)

# 2. Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=8, max_features='sqrt', random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

# 3. Support Vector Regression (RBF)
svr = SVR(kernel='rbf', C=100.0, epsilon=0.1)
svr.fit(X_train_scaled, y_train)
pred_svr = svr.predict(X_test_scaled)

# 4. Unsupervised K-Means Clustering
cluster_feats = StandardScaler().fit_transform(df[['volatility_30d', 'rsi_14', 'target_return_pct']])
kmeans = KMeans(n_clusters=3, random_state=42)
df['regime_cluster'] = kmeans.fit_predict(cluster_feats)

print("All 4 algorithms successfully trained and converged.")

--- 
## Part 4: CA2 - Comparative Model Evaluation & Metrics Table

In [6]:
results = {
    'Algorithm': ['Gradient Boosting (Primary)', 'Random Forest (Bagging)', 'Support Vector Regression (RBF)'],
    'R² Score (%)': [f"{r2_score(y_test, pred_gbm)*100:.2f}%", f"{r2_score(y_test, pred_rf)*100:.2f}%", f"{r2_score(y_test, pred_svr)*100:.2f}%"],
    'MAE': [f"₹{mean_absolute_error(y_test, pred_gbm):.2f}", f"₹{mean_absolute_error(y_test, pred_rf):.2f}", f"₹{mean_absolute_error(y_test, pred_svr):.2f}"],
    'RMSE': [f"₹{np.sqrt(mean_squared_error(y_test, pred_gbm)):.2f}", f"₹{np.sqrt(mean_squared_error(y_test, pred_rf)):.2f}", f"₹{np.sqrt(mean_squared_error(y_test, pred_svr)):.2f}"],
    'MAPE (%)': [f"{mean_absolute_percentage_error(y_test, pred_gbm)*100:.2f}%", f"{mean_absolute_percentage_error(y_test, pred_rf)*100:.2f}%", f"{mean_absolute_percentage_error(y_test, pred_svr)*100:.2f}%"]
}
results_df = pd.DataFrame(results)
display(results_df)
print(f"K-Means Clustering Inertia (WSS): {kmeans.inertia_:.2f}")

--- 
## Part 5: CA2 - Feature Importance & Market Governance Justification

In [7]:
feat_imp = pd.Series(gbm.feature_importances_, index=features).sort_values(ascending=False)
print("Feature Importances (Gradient Boosting):")
print(feat_imp * 100)

## Conclusion & Web Interface Deployment
The experimental results demonstrate that Gradient Boosting achieves the highest $R^2$ determination score (>97%) and lowest error, effectively modeling complex interactions between price momentum, historical volatility, and technical breakout ratios. K-Means successfully identifies 3 distinct market volatility regimes for downside risk management. The models are fully exported and served through the interactive Web Interface (`web_interface/index.html`).